In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import pandas as pd
import xskillscore as xs

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import warnings
import logging

from pathlib import Path

import albedo_functions as af

In [ ]:
# Configura il logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
#        logging.StreamHandler()  # Still see some output in Jupyter
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
def main_script(var, era_var, dset_ctrl, dset_sens, era, save_path, lead, season):
    """Main processing script for computing model acc against ERA5."""
    try:
        logging.info(f"Starting lead {lead} {season}")
        
        # Compute global mean
        dset_ctrl_global = af.global_average(dset_ctrl[var]).to_dataset(name = var)
        dset_sens_global = af.global_average(dset_sens[var]).to_dataset(name = var)
        era_global = af.global_average(era[era_var]).to_dataset(name = var)

        #compute land mean
        dset_ctrl_land = af.global_average(af.land_seas_mask(dset_ctrl[var], 'land')).to_dataset(name = var)
        dset_sens_land = af.global_average(af.land_seas_mask(dset_sens[var], 'land')).to_dataset(name = var)
        era_land = af.global_average(af.land_seas_mask(era[era_var], 'land')).to_dataset(name = var)

        #compute ocean mean
        dset_ctrl_ocean = af.global_average(af.land_seas_mask(dset_ctrl[var], 'ocean')).to_dataset(name = var)
        dset_sens_ocean = af.global_average(af.land_seas_mask(dset_sens[var], 'ocean')).to_dataset(name = var)
        era_ocean = af.global_average(af.land_seas_mask(era[era_var], 'ocean')).to_dataset(name = var)

        logging.info(f"Starting anomalie")
        # Calcolo anomalie
        dset_ctrl_anom = dset_ctrl_global - dset_ctrl_global.mean('time')
        dset_sens_anom = dset_sens_global - dset_sens_global.mean('time')
        
        dset_ctrl_land_anom = dset_ctrl_land - dset_ctrl_land.mean('time')
        dset_sens_land_anom = dset_sens_land - dset_sens_land.mean('time')
        
        dset_ctrl_ocean_anom = dset_ctrl_ocean - dset_ctrl_ocean.mean('time')
        dset_sens_ocean_anom = dset_sens_ocean - dset_sens_ocean.mean('time')
       
        era_anom = era_global - era_global.mean('time')
        era_land_anom = era_land - era_land.mean('time')
        era_ocean_anom = era_ocean - era_ocean.mean('time')    
        
        logging.info(f"Starting correlazioni")
        # Calcolo correlazioni
        em_corr_ctrl = xs.pearson_r(era_anom, dset_ctrl_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl = xs.pearson_r(era_anom, dset_ctrl_anom,  dim='time', skipna=True)
        em_corr_sens = xs.pearson_r(era_anom, dset_sens_anom.mean('member'), dim='time', skipna=True)
        corr_sens = xs.pearson_r(era_anom, dset_sens_anom,  dim='time', skipna=True)

        em_corr_land_ctrl = xs.pearson_r(era_land_anom, dset_ctrl_land_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl_land = xs.pearson_r(era_land_anom, dset_ctrl_land_anom,  dim='time', skipna=True)
        em_corr_land_sens = xs.pearson_r(era_land_anom, dset_sens_land_anom.mean('member'), dim='time', skipna=True)
        corr_sens_land = xs.pearson_r(era_land_anom, dset_sens_land_anom,  dim='time', skipna=True)

        em_corr_ocean_ctrl = xs.pearson_r(era_ocean_anom, dset_ctrl_ocean_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl_ocean = xs.pearson_r(era_ocean_anom, dset_ctrl_ocean_anom,  dim='time', skipna=True)
        em_corr_ocean_sens = xs.pearson_r(era_ocean_anom, dset_sens_ocean_anom.mean('member'), dim='time', skipna=True)
        corr_sens_ocean = xs.pearson_r(era_ocean_anom, dset_sens_ocean_anom,  dim='time', skipna=True)

        # Save results
        ctrl_em_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_{season}_1999.nc'
        ctrl_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_{season}_1999.nc'
        sens_em_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_{season}_1999.nc'
        sens_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_{season}_1999.nc'

        ctrl_em_land_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_land_{season}_1999.nc'
        ctrl_land_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_land_{season}_1999.nc'
        sens_em_land_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_land_{season}_1999.nc'
        sens_land_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_land_{season}_1999.nc'

        ctrl_em_ocean_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_ocean_{season}_1999.nc'
        ctrl_ocean_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_ocean_{season}_1999.nc'
        sens_em_ocean_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_ocean_{season}_1999.nc'
        sens_ocean_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_ocean_{season}_1999.nc'

        em_corr_ctrl.to_netcdf(ctrl_em_outfile)
        corr_ctrl.to_netcdf(ctrl_outfile)
        em_corr_sens.to_netcdf(sens_em_outfile)
        corr_sens.to_netcdf(sens_outfile)

        em_corr_land_ctrl.to_netcdf(ctrl_em_land_outfile)
        corr_ctrl_land.to_netcdf(ctrl_land_outfile)
        em_corr_land_sens.to_netcdf(sens_em_land_outfile)
        corr_sens_land.to_netcdf(sens_land_outfile)
        
        em_corr_ocean_ctrl.to_netcdf(ctrl_em_ocean_outfile)
        corr_ctrl_ocean.to_netcdf(ctrl_ocean_outfile)
        em_corr_ocean_sens.to_netcdf(sens_em_ocean_outfile)
        corr_sens_ocean.to_netcdf(sens_ocean_outfile) 
        
        logging.info(f"Bias files written successfully for lead {lead} {season}")
    
    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
def run_main_script_for_season_and_lead(save_path, season, lead):
    logging.info(f"Starting lead {lead}")
    try:
        # Load control experiment dataset
        ctrl_path = f"{save_path}/{exp_ctrl}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_lead{lead}_rad_{season}.nc"
        dset_ctrl = xr.open_mfdataset(ctrl_path, concat_dim='member', combine='nested', chunks='auto').load()
        dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).to_period('M').to_timestamp()
    
        # Load sensitivity experiment dataset
        sens_path = f"{save_path}/{exp_sens}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_lead{lead}_rad_{season}.nc"
        dset_sens = xr.open_mfdataset(sens_path, concat_dim='member', combine='nested', chunks='auto').load()
        dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).to_period('M').to_timestamp()
        
        # Load ERA5 for the specific season inside the process
        era_path = f"{WORK_DIR}/ERA5_{era_var}_1x1_{season}.nc"
        era = xr.open_dataset(era_path, chunks='auto').load()
        era = era.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))

        #select time from 1999
        start_date = '1999-09-01'
        
        dset_ctrl = dset_ctrl.sel(time=slice(start_date, None))
        dset_sens = dset_sens.sel(time=slice(start_date, None))
        era = era.sel(time=slice(start_date, None))
        
        main_script(var, era_var, dset_ctrl, dset_sens, era, SAVE_PATH, lead, season)
    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
%%time
seasons = ['DJF', 'MAM', 'JJA', 'SON']
futures = []

# Submit all (season, lead) combinations in parallel
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [
        executor.submit(run_main_script_for_season_and_lead, SAVE_PATH, season, lead)
        for season in seasons
        for lead in leads
    ]
    concurrent.futures.wait(futures)